# Mahananda River Survey — Image Detection Notebook

This notebook goes through every photo in your survey folder and:

1. **Object detection (YOLOv8, pretrained)** — finds people, animals (cow, dog, bird, etc.), boats, vehicles, and a few litter-like items (bottle, cup, backpack, etc.), drawn as **green** boxes (red for litter-proxy items).
2. **Land-cover composition** — estimates the % of each photo that is vegetation, water, sky, and "other" (bank/mud/structures), using colour analysis.
3. **Water tint note** — a rough colour-based read on the water (greenish → possible algae, brownish → high turbidity/sediment, greyish → clearer).
4. **Possible-debris flags (heuristic, orange boxes)** — patches of water/bank whose colour stands out sharply from their surroundings. **This is a heuristic proxy, not a trained trash detector** — treat every orange box as "worth a human look," not a confirmed detection.

Everything is saved to your output folder:
- `detected_<filename>.jpg` — annotated copy of each photo
- `summary_report.csv` — one row per photo with all the numbers above
- `landcover_composition.png` and `debris_flags_by_image.png` — quick-look charts across the whole survey

---
### Step 0 — Install the required libraries

Open **Command Prompt** on Windows and run:

```
pip install ultralytics opencv-python pandas matplotlib pillow
```

This pulls in `ultralytics` (YOLOv8 + PyTorch), OpenCV, pandas, matplotlib and Pillow. The first install can take a few minutes and a couple of GB of disk space (PyTorch is the biggest piece). You need an internet connection the *first* time you run the notebook too, because the YOLOv8 model weights (~22 MB) auto-download from GitHub on first use.

If you'd rather install from inside Jupyter, the code cell right below does the same thing — just run it once.

In [ ]:
# Optional: run this once if you didn't already install via Command Prompt.
# Safe to re-run — pip will just say "already satisfied" for anything installed.
# import sys
# !{sys.executable} -m pip install ultralytics opencv-python pandas matplotlib pillow

### Step 1 — Imports

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

print("Libraries loaded OK")

### Step 2 — Set your folders

`INPUT_DIR` is scanned automatically for every `.jpg` / `.jpeg` / `.png` file, so you don't need to paste each filename by hand — any new photos you drop into that folder later will be picked up too.

In [ ]:
INPUT_DIR  = r"D:\Detected_Videos\Mahahanda_River"
OUTPUT_DIR = r"D:\Detected_Videos\MH_dETECTION"

os.makedirs(OUTPUT_DIR, exist_ok=True)

exts = ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.png", "*.PNG")
image_paths = sorted(set(p for ext in exts for p in glob.glob(os.path.join(INPUT_DIR, ext))))

print(f"Found {len(image_paths)} images in {INPUT_DIR}")
for p in image_paths[:5]:
    print(" -", os.path.basename(p))
if len(image_paths) > 5:
    print(f"   ... and {len(image_paths) - 5} more")

### Step 3 — Load the object detector

`yolov8s.pt` is a general-purpose detector trained on the COCO dataset (80 everyday object classes: people, animals, vehicles, bottles, etc). It has **no dedicated "pollution" or "sewage" class** — nobody publishes one, because those aren't visually consistent categories a generic model can learn. That's why this notebook pairs it with the colour-based heuristics below to approximate that side of things.

In [ ]:
model = YOLO("yolov8s.pt")  # auto-downloads the first time you run this

LITTER_PROXY_CLASSES = {"bottle", "cup", "backpack", "handbag", "suitcase",
                         "sports ball", "frisbee", "wine glass", "bowl"}

def run_object_detection(img_bgr, conf=0.18):
    results = model.predict(img_bgr, conf=conf, verbose=False)[0]
    detections = []
    for box in results.boxes:
        cls_id = int(box.cls[0])
        name = model.names[cls_id]
        confidence = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        detections.append({
            "class": name, "confidence": round(confidence, 2),
            "box": (x1, y1, x2, y2),
            "is_litter_proxy": name in LITTER_PROXY_CLASSES
        })
    return detections

print("Model ready. Classes it can detect:", list(model.names.values()))

### Step 4 — Land-cover composition + water-tint heuristic

In [ ]:
def analyze_composition(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    total_px = img_bgr.shape[0] * img_bgr.shape[1]

    veg_mask = cv2.inRange(hsv, (35, 40, 30), (90, 255, 255))          # vegetation (green)
    sky_mask = cv2.inRange(hsv, (0, 0, 180), (180, 60, 255))           # sky (bright, low sat)
    water_mask = cv2.inRange(hsv, (10, 10, 40), (45, 140, 220))        # murky water/mud band
    water_mask = cv2.bitwise_and(water_mask, cv2.bitwise_not(veg_mask))
    water_mask = cv2.bitwise_and(water_mask, cv2.bitwise_not(sky_mask))

    accounted = cv2.bitwise_or(veg_mask, cv2.bitwise_or(sky_mask, water_mask))
    other_mask = cv2.bitwise_not(accounted)   # bank / mud / buildings / anomalies

    def pct(mask):
        return round(100 * cv2.countNonZero(mask) / total_px, 1)

    water_color_note = "no clear water region detected"
    if cv2.countNonZero(water_mask) > 500:
        b, g, r = cv2.mean(img_bgr, mask=water_mask)[:3]
        if g > r + 15 and g > b + 15:
            water_color_note = "greenish tint (possible algae / organic pollution)"
        elif r > g and r > b and (r - b) > 20:
            water_color_note = "brownish tint (high turbidity / sediment)"
        elif max(b, g, r) - min(b, g, r) < 15:
            water_color_note = "greyish, relatively low colour tint"
        else:
            water_color_note = "mixed murky tint"

    return {
        "vegetation_pct": pct(veg_mask), "water_pct": pct(water_mask),
        "sky_pct": pct(sky_mask), "other_pct": pct(other_mask),
        "water_color_note": water_color_note,
        "water_mask": water_mask, "other_mask": other_mask,
    }

### Step 5 — "Worth a look" debris/litter heuristic *(approximate)*

This flags patches of water or nearby bank whose colour stands out sharply from their surroundings — a proxy for floating plastic, bags, wrappers, etc. It adapts to each photo's own lighting and filters out small speckle (sun-glints, ripples). **It will miss camouflaged debris and can still occasionally flag natural objects (foam, pale mud, glare) — use it to prioritise which photos to eyeball closely, not as a certified trash count.**

In [ ]:
def find_debris_blobs(img_bgr, water_mask, other_mask, min_area_frac=0.00035, max_area_frac=0.015):
    total_px = img_bgr.shape[0] * img_bgr.shape[1]
    min_area = max(80, min_area_frac * total_px)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)

    water_dil = cv2.dilate(water_mask, np.ones((35, 35), np.uint8))
    bank_near_water = cv2.bitwise_and(other_mask, water_dil)
    search_zone = cv2.bitwise_or(water_mask, bank_near_water)

    if cv2.countNonZero(search_zone) < 800:
        return []

    mean, std = cv2.meanStdDev(hsv, mask=search_zone)
    mean = mean.flatten()
    std = np.clip(std.flatten(), 6, None)

    diff = np.abs(hsv - mean) / std
    anomaly_score = diff.max(axis=2)

    anomaly_mask = ((anomaly_score > 2.6).astype(np.uint8)) * 255
    anomaly_mask = cv2.bitwise_and(anomaly_mask, search_zone)
    anomaly_mask = cv2.morphologyEx(anomaly_mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    anomaly_mask = cv2.morphologyEx(anomaly_mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))

    contours, _ = cv2.findContours(anomaly_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    blobs = []
    for c in contours:
        area = cv2.contourArea(c)
        if min_area <= area <= max_area_frac * total_px:
            x, y, w, h = cv2.boundingRect(c)
            if max(w, h) / max(min(w, h), 1) > 8:
                continue
            blobs.append((x, y, w, h, int(area)))

    return sorted(blobs, key=lambda b: -b[4])[:15]

### Step 6 — Put it together: annotate + summarise one image

In [ ]:
def process_image(path, out_dir):
    img = cv2.imread(path)
    if img is None:
        return {"file": os.path.basename(path), "error": "could not read image"}

    detections = run_object_detection(img)
    comp = analyze_composition(img)
    blobs = find_debris_blobs(img, comp["water_mask"], comp["other_mask"])

    annotated = img.copy()
    for d in detections:
        x1, y1, x2, y2 = d["box"]
        color = (0, 0, 255) if d["is_litter_proxy"] else (0, 200, 0)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated, f'{d["class"]} {d["confidence"]}', (x1, max(y1 - 6, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    for (x, y, w, h, area) in blobs:
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 165, 255), 2)
        cv2.putText(annotated, "possible debris", (x, max(y - 6, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 165, 255), 1)

    fname = os.path.basename(path)
    out_path = os.path.join(out_dir, f"detected_{fname}")
    cv2.imwrite(out_path, annotated)

    return {
        "file": fname,
        "objects_detected": ", ".join(f'{d["class"]}({d["confidence"]})' for d in detections) or "none",
        "litter_proxy_objects": sum(1 for d in detections if d["is_litter_proxy"]),
        "vegetation_pct": comp["vegetation_pct"],
        "water_pct": comp["water_pct"],
        "sky_pct": comp["sky_pct"],
        "other_pct": comp["other_pct"],
        "water_color_note": comp["water_color_note"],
        "possible_debris_blobs": len(blobs),
        "output_image": out_path,
    }

### Step 7 — Run it on every image in the folder

In [ ]:
rows = []
for i, path in enumerate(image_paths, 1):
    row = process_image(path, OUTPUT_DIR)
    rows.append(row)
    print(f"[{i}/{len(image_paths)}] {row['file']}  ->  "
          f"objects: {row.get('objects_detected','error')} | "
          f"debris flags: {row.get('possible_debris_blobs','-')}")

df = pd.DataFrame(rows)
csv_path = os.path.join(OUTPUT_DIR, "summary_report.csv")
df.to_csv(csv_path, index=False)
print(f"\nSaved summary for {len(df)} images to: {csv_path}")
df.head(10)

### Step 8 — Quick-look charts across the whole survey

In [ ]:
ok = df[df["file"].notna()].copy()
if "error" in ok.columns:
    ok = ok[ok["error"].isna()] if "error" in ok else ok

# --- Land-cover composition, stacked bar per image
fig, ax = plt.subplots(figsize=(max(8, len(ok) * 0.4), 5))
labels = ok["file"]
bottom = np.zeros(len(ok))
for col, color in [("vegetation_pct", "#4c9a2a"), ("water_pct", "#4a7ebb"),
                    ("sky_pct", "#cfd8dc"), ("other_pct", "#a1887f")]:
    ax.bar(labels, ok[col], bottom=bottom, label=col.replace("_pct", ""), color=color)
    bottom += ok[col].values
ax.set_ylabel("% of frame")
ax.set_title("Land-cover composition per photo")
ax.legend(loc="upper right")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "landcover_composition.png"), dpi=150)
plt.show()

# --- Debris flags per image
fig, ax = plt.subplots(figsize=(max(8, len(ok) * 0.4), 4))
ax.bar(ok["file"], ok["possible_debris_blobs"], color="#e08a2e")
ax.set_ylabel("Possible-debris flags")
ax.set_title("Possible-debris flags per photo (heuristic — verify visually)")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "debris_flags_by_image.png"), dpi=150)
plt.show()

print("Charts saved to", OUTPUT_DIR)

### Notes & limitations

- **Reliable:** the YOLOv8 object boxes (person/animal/vehicle/boat/bottle-type items) — this is a real trained detector.
- **Approximate:** vegetation/water/sky/other %, the water-tint note, and the orange "possible debris" boxes. These are colour-based heuristics tuned on your sample photos, not trained models — good for flagging photos to review by eye and for spotting broad trends across the survey, not for a defensible pollution count.
- If you want a genuinely trained litter/trash detector down the line, that needs a labelled dataset of river-litter photos (e.g. the public **TACO** dataset, or your own photos labelled with a tool like Roboflow) and a short fine-tuning run on top of YOLOv8 — happy to help set that up if useful.

### Where to find your results
- Annotated photos: `D:\Detected_Videos\MH_dETECTION\detected_*.jpg`
- Full numbers: `D:\Detected_Videos\MH_dETECTION\summary_report.csv`
- Charts: `landcover_composition.png`, `debris_flags_by_image.png`